In [8]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings('ignore')

def refined_engineering(df):
    df['record_date'] = pd.to_datetime(df['record_date'], format='mixed')
    
    # 1. Tempo e Ciclos
    df['hour'] = df['record_date'].dt.hour
    df['day_week'] = df['record_date'].dt.dayofweek
    df['month'] = df['record_date'].dt.month
    
    # Seno/Cosseno para a hora (faz com que 23h seja perto de 00h)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour']/23.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour']/23.0)
    
    # 2. Engenharia de Tráfego Avançada
    # A métrica de ouro: Atraso por km (aproximado)
    df['relative_delay'] = df['AVERAGE_TIME_DIFF'] / (df['AVERAGE_FREE_FLOW_TIME'] + 1)
    
    # Interação entre Velocidade Livre e Tempo (Capacidade da via)
    df['road_capacity_index'] = df['AVERAGE_FREE_FLOW_SPEED'] * df['AVERAGE_FREE_FLOW_TIME']
    
    # 3. Limpeza de Clima (V3)
    def simplify_cloud(x):
        x = str(x).lower()
        if 'claro' in x or 'limpo' in x: return 0
        if 'nuvens' in x or 'nublado' in x: return 1
        return 0.5 # Para valores nulos ou outros
    
    df['cloud_score'] = df['AVERAGE_CLOUDINESS'].apply(simplify_cloud)
    
    # 4. Luminosidade Binária
    df['is_dark'] = (df['LUMINOSITY'] == 'DARK').astype(int)
    
    # 5. Dash de Features (interações não-lineares)
    df['time_speed_interaction'] = df['AVERAGE_TIME_DIFF'] * df['AVERAGE_FREE_FLOW_SPEED']

    # Drops
    drop_cols = ['city_name', 'record_date', 'AVERAGE_PRECIPITATION', 'AVERAGE_RAIN', 'LUMINOSITY', 'AVERAGE_CLOUDINESS']
    return df.drop(columns=[c for c in drop_cols if c in df.columns])

# --- DATA LOAD ---
train = pd.read_csv("training_data.csv", encoding="latin-1")
test = pd.read_csv("test_data.csv", encoding="latin-1")

train['AVERAGE_SPEED_DIFF'] = train['AVERAGE_SPEED_DIFF'].fillna('None')

# LabelEncoder
le = LabelEncoder()
y = le.fit_transform(train['AVERAGE_SPEED_DIFF'])

X = refined_engineering(train.drop(columns=['AVERAGE_SPEED_DIFF']))
X_test = refined_engineering(test)

# --- MODELAGEM ELITE (VOTING SOFT) ---
# Em vez de Stacking, usamos Voting Soft com pesos. É menos propenso a overfitting.

clf1 = LGBMClassifier(n_estimators=1200, learning_rate=0.015, num_leaves=45, max_depth=9, 
                      subsample=0.8, colsample_bytree=0.7, random_state=42, verbosity=-1)

clf2 = XGBClassifier(n_estimators=1200, learning_rate=0.015, max_depth=8, 
                      subsample=0.8, colsample_bytree=0.7, random_state=42)

clf3 = CatBoostClassifier(iterations=1200, learning_rate=0.015, depth=8, 
                          l2_leaf_reg=5, random_state=42, verbose=0)

# O segredo do 0.84+ está aqui: Pesos desequilibrados
# O CatBoost costuma ser mais conservador e melhor no Leaderboard privado
voter = VotingClassifier(
    estimators=[('lgb', clf1), ('xgb', clf2), ('cat', clf3)],
    voting='soft',
    weights=[1, 1, 1.5] 
)

# --- VALIDAÇÃO ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(voter, X, y, cv=cv, scoring='accuracy')
print(f"Predição de Accuracy Local (CV): {np.mean(scores):.5f}")

# --- FINAL FIT & SUBMIT ---
voter.fit(X, y)
preds = voter.predict(X_test)
labels = le.inverse_transform(preds)

submission = pd.DataFrame({
    "RowId": np.arange(1, len(test) + 1),
    "Speed_Diff": [str(p).replace('_', ' ').title().replace(' ', '_') for p in labels]
})
# Correção final de nomenclatura
submission['Speed_Diff'] = submission['Speed_Diff'].replace('None', 'None')

submission.to_csv("submission_elite_v5.csv", index=False)
print("Ficheiro v5 gerado!")

Predição de Accuracy Local (CV): 0.80769
Ficheiro v5 gerado!
